# Chuẩn bị dữ liệu và code đối chứng

Dùng trước notebook 07/08. Dữ liệu CTT có timestamp đi lùi; phép so dùng **content-only theo thứ tự file**, raw 28/structural 30 chiều. Giữ nguyên dữ liệu gốc. Đọc `code/comparison/README.txt`.

Trên Colab: đặt dự án trong Drive, chọn GPU; đưa archive `ctt_comparison_data_v1.zip` vào `code/output/` nếu dùng dữ liệu đã chuẩn bị. Không cần tải ROAD/SynCAN cho năm mô hình trong phép so này.

In [1]:
from pathlib import Path
import os, subprocess, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
PROJECT_HINT = Path('/content/drive/MyDrive/00_Research')  # sửa nếu đặt dự án nơi khác
candidates = [PROJECT_HINT, Path.cwd(), *Path.cwd().parents]
CODE = next((p / 'code' if (p / 'code/comparison/config.json').exists() else p
             for p in candidates if (p / 'code/comparison/config.json').exists() or
             (p / 'comparison/config.json').exists()), None)
assert CODE is not None, 'Đặt PROJECT_HINT tới thư mục chứa code/'
CODE = CODE.resolve()
CACHE = Path('/content/ctt_compare_cache') if IN_COLAB else CODE / 'results/comparison_data_v1'
OUTPUT = CODE / 'results/comparison_content_v1'
PILOT = CODE / 'results/comparison_pilot'
print('Code:', CODE, '\nCache:', CACHE, '\nResults:', OUTPUT)
def run_stage(stage, output=OUTPUT, extra=()):
    subprocess.run([sys.executable, str(CODE/'scripts/run_comparison.py'), '--stage', stage,
                    '--cache', str(CACHE), '--output', str(output), *map(str, extra)], check=True)


Code: F:\00_Research\code 
Cache: F:\00_Research\code\results\comparison_data_v1 
Results: F:\00_Research\code\results\comparison_content_v1


In [2]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(CODE/'requirements.txt')], check=True)
import torch
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU: nên dùng GPU cho full run')

PyTorch: 2.15.0+cpu CUDA: False
CPU: nên dùng GPU cho full run


In [3]:
run_stage('setup')  # tải sparse checkout đúng commit, kiểm tra SHA-256

## Dữ liệu đã chuẩn bị hoặc CTT gốc
Archive chứa cache của toàn bộ file, không phải một phần đầu bản ghi. Nếu không có cache/archive thì cell sau quét CTT gốc trong code/data/. Giữ raw gốc trên Drive; cache local Colab dùng cho tốc độ đọc.

In [4]:
import json, zipfile
ARCHIVE = CODE / 'output/ctt_comparison_data_v1.zip'
if not (CACHE/'manifest.json').exists() and ARCHIVE.exists():
    CACHE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ARCHIVE) as z:
        for info in z.infolist():
            dest = (CACHE/info.filename).resolve()
            assert dest.is_relative_to(CACHE.resolve()), 'Unsafe archive member'
            z.extract(info, CACHE)
if not (CACHE/'manifest.json').exists():
    run_stage('prepare', extra=('--root', CODE))
else:
    current = json.loads((CACHE/'manifest.json').read_text())
    if current.get('status') != 'COMPLETE':
        run_stage('prepare', extra=('--root', CODE))
# Full cache checksums, even when original files are not mounted here.
sys.path[:0] = [str(CODE), str(CODE/'src')]
from comparison.data import load_manifest
manifest = load_manifest(CACHE, verify=True)
print('Files:', len(manifest['files']))
print('Frames:', sum(e['frames'] for e in manifest['files']))
print('Timestamp inversions:', sum(e['timestamp_inversions'] for e in manifest['files']))
print('Content-only file-order comparison; timing is NOT repaired.')

Files: 236
Frames: 193240845
Timestamp inversions: 12527
Content-only file-order comparison; timing is NOT repaired.


In [5]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', str(CODE/'tests/test_comparison.py')], cwd=CODE.parent, check=True)

CompletedProcess(args=['f:\\00_Research\\.venv\\Scripts\\python.exe', '-m', 'pytest', '-q', 'F:\\00_Research\\code\\tests\\test_comparison.py'], returncode=0)

Tiếp tục notebook 07. Giữ `manifest.json`; thay code/config/data cần output mới. Notebook không tự đưa kết quả pilot vào báo cáo nghiên cứu.